# Ignite2025 Demo Data Generator (Fabric / Lakehouse)

Generates a reduced-scale, richly annotated synthetic automotive dataset for Agent Council demos and Microsoft Fabric Data Agents. This notebook focuses on fast generation (< ~3 min) while preserving narrative anomalies (campaign uplift, logistics delay) and adding semantic/context columns for natural language querying. No mode switch—demo scale only.

**Key Goals**:
- Smaller but expressive dataset (≈350K sales rows)
- Deterministic reproducibility (hash + fixed seeds)
- Human-friendly table & column names (no views required)
- Pre-aggregated KPI tables for simpler NL questions
- Example question-answer pairs to assist Fabric Data Agent configuration
- Semantic document table for retrieval experiments

**Tables Produced** (Delta):
`dates`, `business_events`, `models`, `dealers`, `customers`, `incentives`, `regional_market_signals`, `inventory`, `car_sales`, `dealer_performance_monthly`, `model_performance_monthly`, `customer_feedback`, `semantic_documents`, `business_questions_answers`, `table_metadata`, `agent_instructions`.

Run top to bottom. Each generation cell prints row counts. After completion you can register tables or directly attach them for Fabric Data Agent ingestion.

## 1. Configuration & Scale
Demo-only constants chosen for quick generation while retaining anomaly signal strength.

In [ ]:
# Install required libraries
%pip install faker

In [ ]:
# Imports & Spark Session Setup
import os, hashlib, random, math, pandas as pd
from datetime import datetime, timedelta
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

SEED = 20251110
Faker.seed(SEED); random.seed(SEED); faker = Faker()
try:
    spark
    print('✅ Using existing Spark session')
except NameError:
    spark = (SparkSession.builder
              .appName('Ignite2025 Demo Data Generator')
              .config('spark.sql.extensions','io.delta.sql.DeltaSparkSessionExtension')
              .config('spark.sql.catalog.spark_catalog','org.apache.spark.sql.delta.catalog.DeltaCatalog')
              .getOrCreate())
print('Spark version:', spark.version)

In [ ]:
# Demo Scale Constants (no full mode)
NUM_DEALERS = 120
NUM_MODELS = 180
NUM_CUSTOMERS = 15_000
SALES_TARGET_ROWS = 350_000  # ~0.35M fact rows
MONTHS = [f'{y}-{m:02d}' for y in [2024,2025] for m in range(1,13) if not (y==2024 and m < 7)]  # Jul 2024 - Dec 2025
CAMPAIGN_MONTHS = ['2025-06','2025-07','2025-08','2025-09']  # We will include June derived events
LOGISTICS_DELAY_MONTHS = ['2025-08','2025-09']
ANOMALY_BRAND = 'Azure Motors'
ANOMALY_BODYTYPE = 'EV SUV'
REGIONS = ['Northeast','Mid-Atlantic','Midwest','South','Mountain','West','Pacific NW','Southwest']
TABLES_PATH = 'Tables'
os.makedirs(TABLES_PATH, exist_ok=True)
print('✅ Config loaded: months', len(MONTHS))

In [ ]:
# Helper Functions
def hash_random(*items):
    hv = int(hashlib.md5('|'.join(map(str,items)).encode()).hexdigest(),16)
    return (hv % 1_000_000) / 1_000_000.0

def write_delta(df, name, mode='overwrite'):
    path=f'{TABLES_PATH}/{name}'
    df.write.format('delta').mode(mode).save(path)
    print(f'   ✅ {name}: {df.count():,} rows')
    return path

def marketing_desc(brand, body_type):
    tone = ['dynamic','premium','efficient','advanced','versatile'][int(hash_random(brand, body_type)*5) % 5]
    return f'{brand} {body_type} line offers {tone} design focused on customer experience.'

def dealer_profile(city, region, tier):
    return f'{city} dealer serving {region} region with {tier} tier service and community focus.'

print('✅ Helpers ready')

## 2. Dates Dimension (`dates`)
Provides calendar attributes and period flags for campaign & logistics delay reasoning.

In [ ]:
dates_rows = []
for ym in MONTHS:
    y, m = ym.split('-')
    first_day = f'{ym}-01'
    is_campaign_period = ym in CAMPAIGN_MONTHS
    is_delay_month = ym in LOGISTICS_DELAY_MONTHS
    fiscal_qtr = f'Q{((int(m)-1)//3)+1}'
    dates_rows.append({'month': ym, 'first_day': first_day, 'year': int(y), 'month_num': int(m), 'fiscal_qtr': fiscal_qtr, 'is_campaign_period': is_campaign_period, 'is_delay_month': is_delay_month})
dates_df = spark.createDataFrame(dates_rows)
write_delta(dates_df, 'dates')

## 3. Business Events (`business_events`)
Campaign and logistics disruption windows for narrative Q&A.

In [ ]:
events = []
events.append({'event_id':'EVT001','event_type':'campaign','start_month':'2025-06','end_month':'2025-09','affected_regions':'ALL','severity_level':3,'narrative_text':'Azure Motors EV SUV promotion with rebates + low APR boosting volume.'})
events.append({'event_id':'EVT002','event_type':'logistics_delay','start_month':'2025-08','end_month':'2025-09','affected_regions':'West,Pacific NW','severity_level':4,'narrative_text':'Carrier constraints reduce incoming inventory ~30% in West & Pacific NW.'})
events_df = spark.createDataFrame(events)
write_delta(events_df, 'business_events')

## 4. Models (`models`)
Includes marketing descriptions and a campaign relevance flag.

In [ ]:
body_types = ['Sedan','Hatchback','SUV','EV Sedan','EV SUV','Pickup','Van','Coupe','Wagon','Crossover']
brands = [ANOMALY_BRAND]
while len(brands) < 30:
    b = f'{faker.company()} Motors'.replace(',','')
    if b not in brands: brands.append(b)
models = []
# Ensure core anomaly segment presence
for i in range(40):
    trim = f'{faker.color_name()} {(i%12)+1}'
    models.append({
        'model_id': f'M{i:05d}',
        'brand': ANOMALY_BRAND,
        'model_name': f"{ANOMALY_BRAND.split()[0]} {ANOMALY_BODYTYPE} {trim}",
        'body_type': ANOMALY_BODYTYPE,
        'segment': ANOMALY_BODYTYPE,
        'msrp': 25000+(i%90)*500,
        'campaign_relevance_flag': True,
        'model_marketing_desc': marketing_desc(ANOMALY_BRAND, ANOMALY_BODYTYPE)
    })
for i in range(40, NUM_MODELS):
    brand = brands[i % len(brands)]
    bt = body_types[i % len(body_types)]
    trim = f'{faker.color_name()} {(i%12)+1}'
    models.append({
        'model_id': f'M{i:05d}',
        'brand': brand,
        'model_name': f"{brand.split()[0]} {bt} {trim}",
        'body_type': bt,
        'segment': bt,
        'msrp': 23000+(i%100)*550,
        'campaign_relevance_flag': (brand==ANOMALY_BRAND and bt==ANOMALY_BODYTYPE),
        'model_marketing_desc': marketing_desc(brand, bt)
    })
models_df = spark.createDataFrame(models)
write_delta(models_df, 'models')

## 5. Dealers (`dealers`)
Dealer profiles with tiering for NL segmentation.

In [ ]:
tiers = ['A','B','C']
dealers = []
for i in range(NUM_DEALERS):
    region = REGIONS[i % len(REGIONS)]
    city = faker.city()
    tier = tiers[i % len(tiers)]
    dealers.append({'dealer_id':f'D{i:05d}', 'dealer_name':f'{city} Auto', 'region':region, 'tier': tier, 'opened_year': 1990 + (i % 30), 'lat':25.0+(i%30)*0.9, 'lon':-124.0+(i%60)*1.2, 'dealer_profile_text': dealer_profile(city, region, tier)})
dealers_df = spark.createDataFrame(dealers)
write_delta(dealers_df, 'dealers')

## 6. Next Steps
Upcoming cells (to be added): customers, incentives, regional_market_signals, inventory, car_sales, KPI tables, feedback, semantic documents, example Q&A, metadata.
You can proceed to add those following the established patterns.

## 6. Customers (`customers`)
Lifecycle segments and tenure for segmentation queries.

In [ ]:
lifecycle_segments = ['New Prospect', 'Active Buyer', 'Loyal Repeat', 'Dormant', 'High Value']
preferred_types = body_types
customers = []
for i in range(NUM_CUSTOMERS):
    segment = lifecycle_segments[int(hash_random(i, 'segment') * len(lifecycle_segments))]
    pref_type = preferred_types[int(hash_random(i, 'pref') * len(preferred_types))]
    tenure_months = 1 + int(hash_random(i, 'tenure') * 120)
    customers.append({
        'customer_id': f'C{i:07d}',
        'first_name': faker.first_name(),
        'last_name': faker.last_name(),
        'email': faker.email().lower(),
        'age': 21 + (i % 60),
        'annual_income': 35000 + (i % 120) * 1200 + (i % 7) * 900,
        'lifecycle_segment': segment,
        'preferred_body_type': pref_type,
        'tenure_months': tenure_months
    })
    if i and i % 5000 == 0:
        print(f'   {i:,} customers...')
customers_df = spark.createDataFrame(customers)
write_delta(customers_df, 'customers')

## 7. Incentives (`incentives`)
Monthly rebates & APR with campaign boost and promotional summaries.

In [ ]:
incentives = []
for brand in brands:
    for bt in body_types:
        for m in MONTHS:
            base_rebate = int(hash_random(brand, bt, m) * 3500)
            base_apr = (hash_random(brand, m) * 5.0) + 0.9
            rebate = base_rebate
            apr = base_apr
            if brand == ANOMALY_BRAND and bt == ANOMALY_BODYTYPE and m in CAMPAIGN_MONTHS:
                rebate += 3000
                apr = 1.9
            promo_text = f'${rebate} rebate at {apr:.1f}% APR'
            if brand == ANOMALY_BRAND and bt == ANOMALY_BODYTYPE and m in CAMPAIGN_MONTHS:
                promo_text += ' (Campaign Special)'
            incentives.append({
                'brand': brand,
                'body_type': bt,
                'month': m,
                'rebate': rebate,
                'apr': apr,
                'promo_summary_text': promo_text
            })
incentives_df = spark.createDataFrame(incentives)
write_delta(incentives_df, 'incentives')

## 8. Regional Market Signals (`regional_market_signals`)
External context: consumer confidence, EV adoption index, weather disruption.

In [ ]:
signals = []
for region in REGIONS:
    for m in MONTHS:
        base_confidence = 60 + hash_random(region, m) * 30
        ev_adoption = 5 + hash_random(region, m, 'ev') * 25
        weather_score = 0
        if region in ['West', 'Pacific NW'] and m in LOGISTICS_DELAY_MONTHS:
            weather_score = 7 + int(hash_random(region, m, 'weather') * 3)
        signals.append({
            'region': region,
            'month': m,
            'consumer_confidence_index': __builtins__.round(base_confidence, 1),
            'ev_adoption_index': __builtins__.round(ev_adoption, 1),
            'weather_disruption_score': weather_score
        })
signals_df = spark.createDataFrame(signals)
write_delta(signals_df, 'regional_market_signals')

## 9. Inventory (`inventory`)
Dealer×Model×Month snapshot with stockout flags and logistics delay impact.

In [ ]:
inventory = []
count = 0
for d in dealers:
    for m in models:
        for month in MONTHS:
            on_hand = 80 + int(hash_random(d['dealer_id'], m['model_id'], month) * 80)
            incoming = 10 + int(hash_random(m['model_id'], month) * 40)
            turn_rate = (hash_random(d['dealer_id'], m['model_id']) * 4.0) + 0.8
            logistics_delay_factor = 1.0
            if d['region'] in ['West', 'Pacific NW'] and month in LOGISTICS_DELAY_MONTHS:
                logistics_delay_factor = 0.7
                incoming = int(incoming * logistics_delay_factor)
            stockout_flag = on_hand < 15 and incoming < 5
            inventory.append({
                'dealer_id': d['dealer_id'],
                'region': d['region'],
                'model_id': m['model_id'],
                'brand': m['brand'],
                'body_type': m['body_type'],
                'month': month,
                'on_hand': on_hand,
                'incoming': incoming,
                'turn_rate': __builtins__.round(turn_rate, 2),
                'logistics_delay_factor': logistics_delay_factor,
                'stockout_flag': stockout_flag
            })
            count += 1
            if count and count % 50000 == 0:
                print(f'   {count:,} inventory rows...')
inventory_df = spark.createDataFrame(inventory)
write_delta(inventory_df, 'inventory')

## 10. Car Sales (`car_sales`)
Main fact table (~350K rows) with quality scores, elasticity buckets, margin %.

In [ ]:
# Build incentives lookup
inc_lookup = {(i['brand'], i['body_type'], i['month']): i for i in incentives}
per_combo = NUM_DEALERS * NUM_MODELS * len(MONTHS)
repeat = __builtins__.max(1, int((SALES_TARGET_ROWS / per_combo) + 0.5))
TARGET = SALES_TARGET_ROWS
BATCH_SIZE = 100_000
sc = 0
batch = []
first_write = True

def flush(batch_rows, first):
    if not batch_rows:
        return first
    df = spark.createDataFrame(batch_rows)
    mode = 'overwrite' if first else 'append'
    write_delta(df, 'car_sales', mode=mode)
    return False

for d in dealers:
    for m in models:
        for month in MONTHS:
            for r in range(repeat):
                if sc >= TARGET:
                    break
                rep = str(r)
                inc = inc_lookup.get((m['brand'], m['body_type'], month), {})
                units = 1 + int(hash_random(d['dealer_id'], m['model_id'], rep) * 4)
                discount = int(hash_random(d['dealer_id'], m['model_id'], month) * 2000)
                day = 1 + int(hash_random(d['dealer_id'], m['model_id'], rep) * 28)
                year, mo = month.split('-')
                sale_date = f'{year}-{mo}-{day:02d}'
                cust_idx = int(hash_random(d['dealer_id'], m['model_id'], rep) * len(customers))
                cust = customers[cust_idx]['customer_id']
                promotion_applied = (m['brand'] == ANOMALY_BRAND and m['body_type'] == ANOMALY_BODYTYPE and month in CAMPAIGN_MONTHS)
                if promotion_applied:
                    units = int(units * 1.35)
                final_price = m['msrp'] - discount - (inc.get('rebate', 0) * 0.7)
                cogs = m['msrp'] * (0.76 + hash_random(m['model_id'], d['dealer_id']) * 0.1)
                revenue = final_price * units
                profit = revenue - (cogs * units)
                margin_pct = (profit / revenue * 100) if revenue > 0 else 0
                quality_score = 0.85 + hash_random(d['dealer_id'], m['model_id'], month) * 0.14
                discount_pct = (discount / m['msrp'] * 100) if m['msrp'] > 0 else 0
                if discount_pct < 5:
                    elasticity_bucket = 'low'
                elif discount_pct < 10:
                    elasticity_bucket = 'medium'
                else:
                    elasticity_bucket = 'high'
                batch.append({
                    'sale_id': f'S{sc:08d}',
                    'dealer_id': d['dealer_id'],
                    'region': d['region'],
                    'model_id': m['model_id'],
                    'brand': m['brand'],
                    'body_type': m['body_type'],
                    'model_name': m['model_name'],
                    'month': month,
                    'sale_date': sale_date,
                    'customer_id': cust,
                    'units': units,
                    'msrp': m['msrp'],
                    'discount': discount,
                    'rebate': inc.get('rebate', 0),
                    'apr': inc.get('apr', 0.0),
                    'final_price': __builtins__.round(final_price, 2),
                    'cogs': __builtins__.round(cogs, 2),
                    'revenue': __builtins__.round(revenue, 2),
                    'profit': __builtins__.round(profit, 2),
                    'margin_pct': __builtins__.round(margin_pct, 2),
                    'quality_score': __builtins__.round(quality_score, 3),
                    'price_elasticity_bucket': elasticity_bucket,
                    'promotion_applied_flag': promotion_applied
                })
                sc += 1
                if sc % 50000 == 0:
                    print(f'   {sc:,} sales rows (cumulative)...')
                if len(batch) >= BATCH_SIZE:
                    first_write = flush(batch, first_write)
                    batch = []
            if sc >= TARGET:
                break
        if sc >= TARGET:
            break
    if sc >= TARGET:
        break
# Flush remaining
first_write = flush(batch, first_write)
print(f'✅ Sales rows written: {sc:,}')

## 11. Dealer Performance Monthly (`dealer_performance_monthly`)
Pre-aggregated KPIs from car_sales for simplified agent queries.

In [ ]:
# Aggregate from car_sales (ensure car_sales is loaded)
sales_df = spark.read.format('delta').load(f'{TABLES_PATH}/car_sales')
dealer_kpi = sales_df.groupBy('dealer_id', 'region', 'month').agg(
    sum('units').alias('total_units'),
    sum('revenue').alias('total_revenue'),
    sum('profit').alias('total_profit'),
    avg('discount').alias('avg_discount'),
    avg('margin_pct').alias('avg_margin_pct')
)
# Add stockout risk score (synthetic deterministic)
dealer_kpi = dealer_kpi.withColumn('stockout_risk_score', 
    (hash('dealer_id', 'month') % 100) / 10.0)
write_delta(dealer_kpi, 'dealer_performance_monthly')

## 12. Model Performance Monthly (`model_performance_monthly`)
Pre-aggregated model KPIs showing campaign uplift signals.

In [ ]:
model_kpi = sales_df.groupBy('model_id', 'brand', 'body_type', 'month').agg(
    sum('units').alias('total_units'),
    avg('final_price').alias('avg_final_price'),
    avg('margin_pct').alias('avg_margin_pct'),
    sum(when(col('promotion_applied_flag') == True, 1).otherwise(0)).alias('promo_units')
)
# Calculate promo uplift %
model_kpi = model_kpi.withColumn('promo_uplift_pct',
    (col('promo_units') / col('total_units') * 100))
write_delta(model_kpi, 'model_performance_monthly')

## 13. Customer Feedback (`customer_feedback`)
Synthetic feedback snippets with sentiment for NL queries.

In [ ]:
sentiments = ['positive', 'neutral', 'negative']
categories = ['pricing', 'availability', 'incentives', 'service', 'quality']
feedback_templates = {
    'positive': ['Great value for the price', 'Excellent customer service', 'Very satisfied with my purchase', 'Incentives made it affordable', 'Highly recommend this model'],
    'neutral': ['Average experience', 'Met expectations', 'Standard process', 'Acceptable wait time', 'Fair pricing'],
    'negative': ['Limited inventory', 'Long wait for delivery', 'Expected better incentives', 'Service could improve', 'Disappointed with availability']
}
feedbacks = []
for i in range(800):
    cust = customers[i % len(customers)]
    model = models[i % len(models)]
    month = MONTHS[i % len(MONTHS)]
    sentiment = sentiments[int(hash_random(i, 'sentiment') * len(sentiments))]
    category = categories[int(hash_random(i, 'category') * len(categories))]
    template_idx = int(hash_random(i, 'template') * len(feedback_templates[sentiment]))
    feedback_text = feedback_templates[sentiment][template_idx]
    feedbacks.append({
        'feedback_id': f'FB{i:06d}',
        'customer_id': cust['customer_id'],
        'model_id': model['model_id'],
        'month': month,
        'sentiment': sentiment,
        'category': category,
        'feedback_text': feedback_text
    })
feedback_df = spark.createDataFrame(feedbacks)
write_delta(feedback_df, 'customer_feedback')

## 14. Semantic Documents (`semantic_documents`)
Denormalized textual rows for retrieval/embedding experiments.

In [ ]:
# Sample 500 semantic documents from aggregated sales
semantic_docs = []
dealer_kpi_list = dealer_kpi.collect()
for i, row in enumerate(dealer_kpi_list[:500]):
    period_type = 'campaign' if row['month'] in CAMPAIGN_MONTHS else 'baseline'
    doc_text = (f"Month {row['month']}: Dealer {row['dealer_id']} in {row['region']} "
                f"sold {row['total_units']} units generating ${row['total_revenue']:,.2f} revenue "
                f"with margin {row['avg_margin_pct']:.1f}% during {period_type} period.")
    semantic_docs.append({
        'doc_id': f'DOC{i:06d}',
        'dealer_id': row['dealer_id'],
        'region': row['region'],
        'month': row['month'],
        'document_text': doc_text
    })
semantic_df = spark.createDataFrame(semantic_docs)
write_delta(semantic_df, 'semantic_documents')

## 15. Business Questions & Answers (`business_questions_answers`)
Curated Q&A pairs to guide Fabric Data Agent query generation (like "example queries").

In [ ]:
qa_pairs = [
    {
        'qa_id': 'QA001',
        'question_text': 'What was the total revenue during the campaign months?',
        'answer_sql': "SELECT SUM(revenue) AS campaign_revenue FROM car_sales WHERE month IN ('2025-06','2025-07','2025-08','2025-09')",
        'answer_summary': 'Total revenue during Jun-Sep 2025 campaign period.',
        'topic_tags': 'campaign,revenue'
    },
    {
        'qa_id': 'QA002',
        'question_text': 'Which dealers had the highest profit last quarter?',
        'answer_sql': "SELECT TOP 10 dealer_id, SUM(total_profit) AS q3_profit FROM dealer_performance_monthly WHERE month IN ('2025-07','2025-08','2025-09') GROUP BY dealer_id ORDER BY q3_profit DESC",
        'answer_summary': 'Top 10 dealers by profit in Q3 2025.',
        'topic_tags': 'dealer,profit'
    },
    {
        'qa_id': 'QA003',
        'question_text': 'How did incoming inventory change in West vs Pacific NW during the delay?',
        'answer_sql': "SELECT region, month, AVG(incoming) AS avg_incoming FROM inventory WHERE region IN ('West','Pacific NW') AND month IN ('2025-08','2025-09') GROUP BY region, month ORDER BY region, month",
        'answer_summary': 'Average incoming inventory for affected regions during logistics delay.',
        'topic_tags': 'logistics,inventory'
    },
    {
        'qa_id': 'QA004',
        'question_text': 'What is the average margin for Azure Motors EV SUVs during campaign months?',
        'answer_sql': "SELECT AVG(margin_pct) AS avg_margin FROM car_sales WHERE brand = 'Azure Motors' AND body_type = 'EV SUV' AND month IN ('2025-06','2025-07','2025-08','2025-09')",
        'answer_summary': 'Average profit margin % for campaign-targeted models.',
        'topic_tags': 'campaign,margin,model'
    },
    {
        'qa_id': 'QA005',
        'question_text': 'Which region had the highest stockout risk in September 2025?',
        'answer_sql': "SELECT TOP 1 region, AVG(stockout_risk_score) AS avg_risk FROM dealer_performance_monthly WHERE month = '2025-09' GROUP BY region ORDER BY avg_risk DESC",
        'answer_summary': 'Region with highest average stockout risk score.',
        'topic_tags': 'inventory,risk,region'
    },
    {
        'qa_id': 'QA006',
        'question_text': 'What is the customer sentiment distribution for EV SUVs?',
        'answer_sql': "SELECT f.sentiment, COUNT(*) AS feedback_count FROM customer_feedback f JOIN models m ON f.model_id = m.model_id WHERE m.body_type = 'EV SUV' GROUP BY f.sentiment ORDER BY feedback_count DESC",
        'answer_summary': 'Count of positive/neutral/negative feedback for EV SUV models.',
        'topic_tags': 'sentiment,customer,model'
    },
    {
        'qa_id': 'QA007',
        'question_text': 'Show total units sold by body type in 2025',
        'answer_sql': "SELECT body_type, SUM(units) AS total_units FROM car_sales WHERE month LIKE '2025-%' GROUP BY body_type ORDER BY total_units DESC",
        'answer_summary': 'Units sold aggregated by vehicle body type for 2025.',
        'topic_tags': 'sales,body_type'
    },
    {
        'qa_id': 'QA008',
        'question_text': 'Which models had the highest promotional uplift percentage?',
        'answer_sql': "SELECT TOP 10 model_id, brand, body_type, AVG(promo_uplift_pct) AS avg_uplift FROM model_performance_monthly WHERE promo_uplift_pct > 0 GROUP BY model_id, brand, body_type ORDER BY avg_uplift DESC",
        'answer_summary': 'Top 10 models by average promotional uplift %.',
        'topic_tags': 'campaign,model,uplift'
    },
    {
        'qa_id': 'QA009',
        'question_text': 'What was the average discount percentage during the campaign?',
        'answer_sql': "SELECT AVG(discount / msrp * 100) AS avg_discount_pct FROM car_sales WHERE month IN ('2025-06','2025-07','2025-08','2025-09') AND msrp > 0",
        'answer_summary': 'Average discount as % of MSRP during campaign period.',
        'topic_tags': 'campaign,pricing,discount'
    },
    {
        'qa_id': 'QA010',
        'question_text': 'List dealers with tier A in the Northeast region',
        'answer_sql': "SELECT dealer_id, dealer_name FROM dealers WHERE tier = 'A' AND region = 'Northeast'",
        'answer_summary': 'Tier A dealers located in Northeast region.',
        'topic_tags': 'dealer,region'
    },
    {
        'qa_id': 'QA011',
        'question_text': 'What is the consumer confidence index for the West region in August 2025?',
        'answer_sql': "SELECT consumer_confidence_index FROM regional_market_signals WHERE region = 'West' AND month = '2025-08'",
        'answer_summary': 'Consumer confidence index value for West in Aug 2025.',
        'topic_tags': 'market,signals,region'
    },
    {
        'qa_id': 'QA012',
        'question_text': 'Show the customer count by lifecycle segment',
        'answer_sql': "SELECT lifecycle_segment, COUNT(*) AS segment_customers FROM customers GROUP BY lifecycle_segment ORDER BY segment_customers DESC",
        'answer_summary': 'Distribution of customers across lifecycle segments.',
        'topic_tags': 'customer,segmentation,distribution'
    }
]
qa_df = spark.createDataFrame(qa_pairs)
write_delta(qa_df, 'business_questions_answers')

## 16. Table Metadata (`table_metadata`)
Describes all tables with synonyms, grain, and example questions for agent context.

In [ ]:
metadata = [
    {'table_name': 'dates', 'friendly_name': 'Calendar Dates', 'synonyms': 'calendar,time periods', 'grain': 'month', 'short_description': 'Date dimension with campaign and delay period flags.', 'example_question': 'Which months were part of the campaign?'},
    {'table_name': 'business_events', 'friendly_name': 'Business Events', 'synonyms': 'events,timeline', 'grain': 'event_id', 'short_description': 'Campaign and logistics disruption events.', 'example_question': 'What events occurred in August 2025?'},
    {'table_name': 'models', 'friendly_name': 'Vehicle Models', 'synonyms': 'models,vehicles,cars', 'grain': 'model_id', 'short_description': 'Vehicle models with brand, body type, MSRP, and campaign relevance.', 'example_question': 'Which models are part of the EV SUV campaign?'},
    {'table_name': 'dealers', 'friendly_name': 'Dealers', 'synonyms': 'dealers,dealerships', 'grain': 'dealer_id', 'short_description': 'Dealer locations with region, tier, and profile.', 'example_question': 'List all dealers in the Pacific NW region.'},
    {'table_name': 'customers', 'friendly_name': 'Customers', 'synonyms': 'customers,buyers,lifecycle segments,segments,high value', 'grain': 'customer_id', 'short_description': 'Customer demographics with lifecycle segment and preferences.', 'example_question': 'How many high-value customers are there?'},
    {'table_name': 'incentives', 'friendly_name': 'Incentives', 'synonyms': 'incentives,promotions,rebates', 'grain': 'brand + body_type + month', 'short_description': 'Monthly rebates and APR by brand and body type.', 'example_question': 'What was the rebate for EV SUVs in June 2025?'},
    {'table_name': 'regional_market_signals', 'friendly_name': 'Market Signals', 'synonyms': 'market signals,economic indicators', 'grain': 'region + month', 'short_description': 'Regional consumer confidence, EV adoption, and weather disruption.', 'example_question': 'What was the consumer confidence in the West during August?'},
    {'table_name': 'inventory', 'friendly_name': 'Inventory', 'synonyms': 'inventory,stock', 'grain': 'dealer_id + model_id + month', 'short_description': 'Monthly inventory levels with stockout flags and logistics delay factors.', 'example_question': 'Which dealers had stockouts in September 2025?'},
    {'table_name': 'car_sales', 'friendly_name': 'Car Sales', 'synonyms': 'sales,transactions', 'grain': 'sale_id', 'short_description': 'Sales transactions with units, revenue, profit, margin, and promotion flags.', 'example_question': 'What was total revenue in Q3 2025?'},
    {'table_name': 'dealer_performance_monthly', 'friendly_name': 'Dealer Performance', 'synonyms': 'dealer performance,dealer KPIs,dealer metrics', 'grain': 'dealer_id + month', 'short_description': 'Monthly dealer KPIs: units, revenue, profit, discount, margin, stockout risk.', 'example_question': 'Which dealers had the highest profit last quarter?'},
    {'table_name': 'model_performance_monthly', 'friendly_name': 'Model Performance', 'synonyms': 'model performance,model KPIs,model metrics', 'grain': 'model_id + month', 'short_description': 'Monthly model KPIs: units, price, margin, promotional uplift.', 'example_question': 'Which models had the highest promotional uplift?'},
    {'table_name': 'customer_feedback', 'friendly_name': 'Customer Feedback', 'synonyms': 'feedback,reviews,voice of customer', 'grain': 'feedback_id', 'short_description': 'Synthetic customer feedback with sentiment and category.', 'example_question': 'What is the sentiment distribution for EV SUVs?'},
    {'table_name': 'semantic_documents', 'friendly_name': 'Semantic Documents', 'synonyms': 'documents,text corpus', 'grain': 'doc_id', 'short_description': 'Denormalized textual summaries for retrieval and embedding.', 'example_question': 'Find text describing dealer performance in campaign months.'},
    {'table_name': 'business_questions_answers', 'friendly_name': 'Example Q&A', 'synonyms': 'QA,examples,reference queries', 'grain': 'qa_id', 'short_description': 'Curated question-SQL pairs to guide agent query generation.', 'example_question': 'Show me example questions about the campaign.'},
    {'table_name': 'table_metadata', 'friendly_name': 'Table Metadata', 'synonyms': 'metadata,data dictionary', 'grain': 'table_name', 'short_description': 'Describes all tables with synonyms and example questions.', 'example_question': 'What tables are available?'},
    {'table_name': 'agent_instructions', 'friendly_name': 'Agent Instructions', 'synonyms': 'instructions,guidance', 'grain': 'instruction_id', 'short_description': 'Organizational guidelines for agent query preferences.', 'example_question': 'What guidance exists for dealer queries?'}
]
metadata_df = spark.createDataFrame(metadata)
write_delta(metadata_df, 'table_metadata')

## 17. Agent Instructions (`agent_instructions`)
Organizational guidelines to help agents choose appropriate tables and patterns.

In [ ]:
instructions = [
    {'instruction_id': 1, 'instruction_text': 'NEVER rename or alter table identifiers when restating a user question. Preserve exact table names (e.g., car_sales, dealer_performance_monthly).'},
    {'instruction_id': 2, 'instruction_text': 'Dealer performance, profit, or trend questions: use dealer_performance_monthly (pre-aggregated KPIs) instead of raw car_sales.'},
    {'instruction_id': 3, 'instruction_text': 'Model analysis, promotional uplift, or campaign effectiveness: use model_performance_monthly (contains promo_uplift_pct and aggregated margins).'},
    {'instruction_id': 4, 'instruction_text': 'Inventory, stockouts, or supply chain: use inventory for detailed on_hand/incoming/stockout_flag; for risk scoring use dealer_performance_monthly.stockout_risk_score.'},
    {'instruction_id': 5, 'instruction_text': 'Customer sentiment or feedback: join customer_feedback f ON f.model_id = models.model_id to segment sentiment by brand/body_type.'},
    {'instruction_id': 6, 'instruction_text': 'Campaign period months: 2025-06 through 2025-09. Filter month IN ("2025-06","2025-07","2025-08","2025-09") OR use dates.is_campaign_period = true.'},
    {'instruction_id': 7, 'instruction_text': 'Campaign focus entities: brand = "Azure Motors" AND body_type = "EV SUV" for uplift and margin comparisons.'},
    {'instruction_id': 8, 'instruction_text': 'Logistics delay context: West and Pacific NW regions affected in Aug-Sep 2025. Use regional_market_signals.weather_disruption_score and inventory.logistics_delay_factor (0.7 ≈ 30% reduction).'},
    {'instruction_id': 9, 'instruction_text': 'Top dealer queries: SELECT TOP N dealer_id, region, SUM(total_profit) ... GROUP BY dealer_id, region ORDER BY SUM(total_profit) DESC (use TOP not LIMIT).'},
    {'instruction_id': 10, 'instruction_text': 'Sentiment distribution pattern: SELECT sentiment, COUNT(*) FROM customer_feedback GROUP BY sentiment ORDER BY COUNT(*) DESC.'},
    {'instruction_id': 11, 'instruction_text': 'Lifecycle segmentation queries: use customers.lifecycle_segment. Distribution: SELECT lifecycle_segment, COUNT(*) FROM customers GROUP BY lifecycle_segment ORDER BY COUNT(*) DESC.'},
    {'instruction_id': 12, 'instruction_text': 'Normalize lifecycle synonyms: "high value"=>"High Value"; "loyal"=>"Loyal Repeat"; "active buyer"=>"Active Buyer"; "new prospect"=>"New Prospect"; "dormant"=>"Dormant".'},
    {'instruction_id': 13, 'instruction_text': 'Key metrics glossary: revenue, profit, margin_pct, units, stockout_risk_score, promo_uplift_pct, avg_discount, total_revenue, total_units.'},
    {'instruction_id': 14, 'instruction_text': 'Region canonical list: Northeast, Mid-Atlantic, Midwest, South, Mountain, West, Pacific NW, Southwest. Use exact casing.'},
    {'instruction_id': 15, 'instruction_text': 'Use business_questions_answers table to mirror SQL patterns (QA references) for similar user intents.'},
    {'instruction_id': 16, 'instruction_text': 'Prefer aggregated tables (dealer_performance_monthly, model_performance_monthly) over car_sales when user asks for trends, rankings, or summaries.'},
    {'instruction_id': 17, 'instruction_text': 'When comparing campaign vs baseline, group by CASE WHEN month IN ("2025-06","2025-07","2025-08","2025-09") THEN "Campaign" ELSE "Baseline" END.'},
    {'instruction_id': 18, 'instruction_text': 'For promotional uplift, reference model_performance_monthly.promo_uplift_pct; filter promo_uplift_pct > 0 for most impacted models.'},
    {'instruction_id': 19, 'instruction_text': 'If user asks for inventory impact during delay, compare incoming between affected regions and unaffected regions for months 2025-08, 2025-09.'},
    {'instruction_id': 20, 'instruction_text': 'Always use TOP N syntax for limiting rows (e.g., TOP 10) to align with SQL Server dialect; avoid LIMIT.'}
]
instructions_df = spark.createDataFrame(instructions)
write_delta(instructions_df, 'agent_instructions')

## 18. Sample Analytical Queries
Demonstrate key insights: campaign effectiveness, logistics impact, dealer concentration, customer segmentation.

### 18.A Campaign Effectiveness
Compare Azure Motors EV SUV sales during campaign vs baseline periods.

In [ ]:
campaign_query = """
SELECT 
  CASE WHEN month IN ('2025-06','2025-07','2025-08','2025-09') THEN 'Campaign' ELSE 'Baseline' END AS period,
  SUM(units) AS total_units,
  SUM(revenue) AS total_revenue,
  AVG(margin_pct) AS avg_margin_pct
FROM car_sales
WHERE brand = 'Azure Motors' AND body_type = 'EV SUV'
GROUP BY CASE WHEN month IN ('2025-06','2025-07','2025-08','2025-09') THEN 'Campaign' ELSE 'Baseline' END
ORDER BY period
"""
campaign_df = spark.sql(campaign_query)
display(campaign_df)
print("\\n📊 Campaign vs Baseline: Azure Motors EV SUV shows volume increase during promotional period with margin trade-off.")

### 18.B Logistics Delay Impact
Incoming inventory reduction in affected regions during disruption months.

In [ ]:
logistics_query = """
SELECT region, month, AVG(incoming) AS avg_incoming, AVG(logistics_delay_factor) AS avg_delay_factor
FROM inventory
WHERE region IN ('West', 'Pacific NW') AND month IN ('2025-08', '2025-09')
GROUP BY region, month
ORDER BY region, month
"""
logistics_df = spark.sql(logistics_query)
display(logistics_df)
print("\\n🚚 Logistics Impact: ~30% reduction in incoming inventory for West/Pacific NW during Aug-Sep 2025.")

### 18.C Top Dealer Performance
Highest profit dealers using pre-aggregated KPI table.

In [ ]:
dealer_query = """
SELECT dealer_id, region, SUM(total_profit) AS ytd_profit
FROM dealer_performance_monthly
WHERE month LIKE '2025-%'
GROUP BY dealer_id, region
ORDER BY ytd_profit DESC
LIMIT 10
"""
dealer_df = spark.sql(dealer_query)
display(dealer_df)
print("\\n🏆 Top 10 Dealers by 2025 YTD profit.")

### 18.D Customer Segmentation
Lifecycle segment distribution.

In [ ]:
segment_query = """
SELECT lifecycle_segment, COUNT(*) AS customer_count, AVG(annual_income) AS avg_income
FROM customers
GROUP BY lifecycle_segment
ORDER BY customer_count DESC
"""
segment_df = spark.sql(segment_query)
display(segment_df)
print("\\n👥 Customer lifecycle segments with counts and average income.")

## 19. Final Validation
Check all generated tables with row counts.

In [ ]:
all_tables = [
    'dates', 'business_events', 'models', 'dealers', 'customers', 'incentives',
    'regional_market_signals', 'inventory', 'car_sales', 'dealer_performance_monthly',
    'model_performance_monthly', 'customer_feedback', 'semantic_documents',
    'business_questions_answers', 'table_metadata', 'agent_instructions'
]
print("\\n" + "="*60)
print("TABLE GENERATION SUMMARY")
print("="*60)
total_rows = 0
for table in all_tables:
    try:
        cnt = spark.read.format('delta').load(f'{TABLES_PATH}/{table}').count()
        print(f'{table:30s} -> {cnt:>10,} rows')
        total_rows += cnt
    except Exception as e:
        print(f'{table:30s} -> MISSING or ERROR')
print("="*60)
print(f'{"TOTAL":30s} -> {total_rows:>10,} rows')
print("="*60)
print("\\n✅ All tables generated successfully!")
print(f"📂 Location: {TABLES_PATH}/")
print("🎯 Ready for Fabric Data Agent ingestion.")

## 20. Agent Usage Guidance
How to use this dataset with Microsoft Fabric Data Agents.

### Fabric Data Agent Setup Steps

1. **Select Core Tables** in your Fabric Data Agent configuration:
   - `car_sales` (main fact)
   - `dealer_performance_monthly` (dealer KPIs)
   - `model_performance_monthly` (model KPIs)
   - `customers`, `dealers`, `models` (dimensions)
   - `inventory`, `incentives`, `dates` (supporting facts)

2. **Add Example Queries** from `business_questions_answers`:
   - Query this table: `SELECT question_text, answer_sql FROM business_questions_answers`
   - Copy validated SQL patterns into agent's example query configuration
   - These guide NL→SQL generation for similar questions

3. **Provide Agent Instructions** from `agent_instructions`:
   - Query: `SELECT instruction_text FROM agent_instructions`
   - Add these as agent instructions to guide table selection and query patterns

4. **Reference Metadata** for semantic understanding:
   - `table_metadata` provides synonyms and example questions
   - Can be queried by agent to understand available tables
   - Use synonyms for better NL matching

5. **Optional Enrichment Tables**:
   - `customer_feedback` for sentiment analysis queries
   - `semantic_documents` for retrieval/RAG experiments
   - `regional_market_signals` for external context correlation
   - `business_events` for timeline narrative

### Example Agent Questions to Test
- "What was total revenue during the campaign?"
- "Which dealers had the highest profit last quarter?"
- "How did logistics delays affect West region inventory?"
- "What is customer sentiment for EV SUVs?"
- "Show me models with highest promotional uplift"
- "Which region had the most stockout risk in September?"

### Deterministic Reproducibility
- Seed: `20251110` ensures identical data on each run
- Use for benchmarking agent accuracy across iterations
- Re-run notebook to regenerate if schema changes needed

---
**Next Steps**: Attach this Lakehouse to your Fabric Data Agent and start asking natural language questions!"